In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(verbose=True)

True

In [2]:
llm = ChatOpenAI(model="gpt-5.4-mini")

message = "In 1 sentence, what does it mean for an AI agent to be autonomous"
reply = llm.invoke(message)
print(reply.content)

An AI agent is autonomous if it can independently perceive its environment, make decisions, and take actions to achieve its goals without needing constant human control.


In [4]:
for chunk in llm.stream("Tell me a two line poem about autonomous agents"):
    print(chunk.content, end="", flush=True)

Autonomous agents roam where algorithms dare,  
Learning in the silence, weaving choices from thin air.

In [6]:
deepseek_llm = ChatOpenAI(
    model=os.getenv("DEEPSEEK_MODEL"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
)

reply = deepseek_llm.invoke("In one sentence, what is LangChain?")
print(reply.content)

LangChain is an open-source framework for building applications powered by large language models by composing prompts, tools, memory, retrieval, and agents into workflows.


In [7]:
messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of France?"),
]

print(llm.invoke(messages).content)

messages_as_dict = [
    {
        "role": "system",
        "content": "You are a terse assistant who answers in exactly five words."
    },
    {
        "role": "user",
        "content": "What is the capital of France?"
    }
]
print(llm.invoke(messages_as_dict).content)

Paris is France’s capital city.
Paris is France’s capital city.


In [14]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOD": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print("name:", get_share_price.name)
print("description:", get_share_price.description)
print("args:", get_share_price.args)
print("called directly:", get_share_price.invoke({"symbol": "AAPL"}))

name: get_share_price
description: Return the current share price for a given ticker symbol.
args: {'symbol': {'title': 'Symbol', 'type': 'string'}}
called directly: 241.5


In [15]:
llm_with_tools = llm.bind_tools([get_share_price])
response = llm_with_tools.invoke("What is the share price of Amazon?")
print("content: ", repr(response.content))
print("tool_calls: ", response.tool_calls)

content:  ''
tool_calls:  [{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_Ug6XydGX5uJeaWBu5QUCdAqT', 'type': 'tool_call'}]


In [16]:
conversation = [HumanMessage("What is the share price of Amazon?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

for call in ai_message.tool_calls:
    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

final = llm_with_tools.invoke(conversation)
print(final.content)

Amazon (AMZN) is **$198.00** per share.


In [18]:
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

structured_llm = llm.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Amazon the technology company")
print(company)
print("Just the ticker:", company.ticker)

name='Amazon' ticker='AMZN' founded_year=1994
Just the ticker: AMZN
